![NWM](img/NWM.png)

# Analyze National Water Model Snow Data at Specific Points of Interest
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org)
Last updated:Oct 16, 2025

**Introduction**: This notebook demonstrates how to perform a direct comparison between modeled and observed SWE at individual SNOTEL sites. It uses National Water Model and Snotel data that is collected in the `01_data_collection.ipynb` notebook.

### 1. Prepare the Python Environment

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [ ]:
# import os
# import sys
# prefix = os.environ['CONDA_PREFIX']
# os.environ['GDAL_DATA'] = os.path.join(prefix, 'share', 'gdal')
# os.environ['PROJ_LIB'] = os.path.join(prefix, 'share', 'proj')

# import dask
# import numpy
# import glob
# import pyproj
# import pandas as pd
# import xarray as xr
# import time
# import fsspec
# import rioxarray
# import hvplot.pandas
# from utils import nwm_utils
# import hvplot.pandas
# import holoviews as hv
# import hvplot.xarray
# hv.extension('bokeh')
# import geopandas as gpd
# import matplotlib.pyplot as plt
# from dask.distributed import Client
# import panel as pn
# pn.extension()

# import warnings
# warnings.filterwarnings("ignore")


import holoviews as hv
import hvplot.pandas

hv.extension('bokeh')

In [ ]:
# Import the Evaluation library from the project root.
from pathlib import Path
sys.path.append(str((Path.cwd().absolute() / "../../../src").resolve()))

from cssi_evaluation.snow import utils

### 2. Build the Input DataFrame

This workflow requires us to organize our observed and modeled snow water equivilent data into a single Pandas DataFrame. We'll do this by reading the data collected in the `01_data_collection.ipynb` notebook.

In [ ]:
# gather all the paths to all of the observation and modeled SWE CSV files
obs = glob.glob(os.path.join('obs_outputs', '*.csv'))
mod = glob.glob(os.path.join('mod_outputs', '*.csv'))

In [ ]:
# Combine these into a single dataframe using a utilitiy function from the nwm_utils library
combined_df = nwm_utils.combine(obs, mod, StartDate, EndDate)
combined_df

### 3. Compare modeled SWE with observed at a site

Plot the GIS data associated with the records in this DataFrame.

In [ ]:
# # Path to the watershed shapefile
watershed = "./domain_data/TolumneRiver_18040009.shp"

# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]
filtered_all_stations_gdf = all_stations_gdf[all_stations_gdf.state.str.contains('California')]  

# Extract the bounding box coordinates of a watershed
watershed_gdf = gpd.read_file(os.path.join(os.getcwd(), watershed)).to_crs(epsg=4326)

# Combine all polygons into a single MultiPolygon
watershed_union = watershed_gdf.geometry.unary_union

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = filtered_all_stations_gdf[filtered_all_stations_gdf.geometry.within(watershed_union)]
gdf_in_bbox.reset_index(inplace=True)
print(f'There are {len(gdf_in_bbox)} sites from', ', '.join(set(gdf_in_bbox.network)), 'network(s) within the watershed.')


m = nwm_utils.plot_sites_within_domain(gdf_in_bbox, watershed_gdf, zoom_start=9)
m

Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Once you’ve identified the site of interest, enter its site code in the next code cell.

In [ ]:
# choose a gage of interest within the watershed
my_site_code = 'HRS'

In [ ]:
gdf_in_bbox[gdf_in_bbox['code']=='HRS']

Plot both datasets.

In [ ]:
nwm_utils.comparison_plots(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

In [ ]:
# combined_df.columns

In [ ]:
# utils.compute_water_year(combined_df, inplace=True)

In [ ]:
## Compute water_year
#combined_df['Water_Year'] = combined_df.index.map(lambda x: x.year+1 if x.month>9 else x.year)

In [ ]:
# combined_df.columns

In [ ]:
# import importlib
# importlib.reload(utils)

Compute the Same-Day SWE Comparison. This is a comparison of the maximum observed SWE for a given year and the corresponding modeled SWE at the time of this occurence. 

**TODO:** Insert a statement explaining this better, it should highligh what this comparison is used for and what it tells us about the modeled data.

In [ ]:
# isolate the columns associated with observations and model predictions.
# these will be inputs to our same-day comparison function.
obs_cols = sorted([col for col in combined_df.columns if col.startswith('CCSS')])
mod_cols = sorted([col for col in combined_df.columns if col.startswith('NWM')])

In [ ]:
# compute the same-day SWE comparison for each of the observed and modeled sites.
df_same_day = utils.same_day_swe_comparison(combined_df, obs_cols, mod_cols)
df_same_day

Visualize these data graphically.

In [ ]:
# Create the scatter plot
scatter_plot_same = df_same_day.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (m)',
    ylabel='Modeled SWE (m)',
    title='SWE on the observed peak SWE date',
    size=15,
    width=400,
    height=300,
    color='#0072B2'
).relabel('Same-Day Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_same_day.drop(columns=['Water_Year']).select_dtypes(include='number').max().max()
one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_same * one_to_one_line).opts(
    legend_position='top'
)

scatter_with_line

Compute the Different-Day SWE Comparison. This is a comparison of the maximum observed and modeled SWEs for a given year and their corresponding dates of occurence. 

**TODO:** Insert a statement explaining this better, it should highligh what this comparison is used for and what it tells us about the modeled data.

In [ ]:
# compute the different-day SWE comparison for each of the observed and modeled sites.
df_diff_day = utils.different_day_swe_comparison(combined_df, obs_cols, mod_cols)
df_diff_day

Visualize these data graphically.

In [ ]:
# Create the scatter plot
scatter_plot_diff = df_diff_day.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (m)',
    ylabel='Modeled SWE (m)',
    title='Peak SWE Magnitude Comparison',
    size=15,
    width=400,
    height=300,
    color='#E69F00'
).relabel('Different-Day Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_diff_day.drop(columns=['Water_Year']).select_dtypes(include='number').max().max()
one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_same * scatter_plot_diff * one_to_one_line).opts(
    legend_position='top'
)

scatter_with_line

In [ ]:
df_diff_day['Peak_Date_Diff_Days'] = (df_diff_day['Modeled_Date'] - 
                                      df_diff_day['Observed_Date']).dt.days
df_diff_day['SWE_Diff'] = (df_diff_day['Modeled'] - 
                           df_diff_day['Observed'])

df_diff_day.hvplot.bar(
    x='Station',
    y='Peak_Date_Diff_Days',
    rot=45,
    ylabel='Date Difference (days)',
    title='Difference Between Observed and Modeled Peak SWE Dates',
    width=700,
    height=400,
    color='Peak_Date_Diff_Days'
)


In [ ]:


scatter = df_diff_day.hvplot.scatter(
    x='Peak_Date_Diff_Days',
    y='SWE_Diff',
    by='Station',
    xlabel='Date Difference (days)',
    ylabel='SWE Difference (m)',
    title='SWE Error vs. Date Shift Between Observed and Modeled Peaks',
    size=80,
    width=600,
    height=400
)

# Add reference lines
vline = hv.VLine(0).opts(color='gray', line_dash='dashed')
hline = hv.HLine(0).opts(color='gray', line_dash='dashed')

(scatter * vline * hline).opts(legend_position='top_left')


Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization helps reveal whether there are **seasonal patterns** in the relationship between observed and modeled SWE. In fact, it allows us to see if the model performs differently during key periods, such as the <span style="color: teal;">**accumulation**</span> phase or the <span style="color: tomato;">**melt**</span> period. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season, which is critical for understanding how well the model represents key processes driving snowpack evolution.

You can change the list of months to highlight (for example, 10 for October or 1 for January) to see how highlighting different parts of the year changes the appearance and interpretation of the scatter plot.

In [ ]:
combined_df['month'] = combined_df.index.month

plot = nwm_utils.plot_custom_scatter(combined_df, my_site_code, highlight_months=[10, 11, 12, 1])
plot

Compute statistics

In [ ]:
nwm_utils.compute_stats(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

Pearson and Spearman correlations are both close to 1, suggesting a strong relationship between observed and modeled SWE. As shown on the timeseries plot, this strong correlation alone does not indicate a "good" model. For example, it does not guarantee accurate timing of key events, such as peak SWE or melt onset. Let's compare these as well. The following code uses `report_max_dates_and_values` function to identify the peak SWE value and the date it occurs for both the observed (CCSS) and modeled (NWM) datasets. 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why? As you review the results, compare the peak flow amounts and the timing of snowmelt onset. Do you see any significant differences? Which dataset indicates an earlier melt?</p>
</div>

In [ ]:
summary_table = nwm_utils.report_max_dates_and_values(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
summary_table

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What is the modeled SWE on the date when the observed SWE reaches its peak?
    Use the code snippet below to find the answer.
  </p>
  <pre><code class="language-python">
  
    # Find date of the peak SWE from observed data
    date_obs_max = combined_df['CCSS_HRS_swe_m'].idxmax()

    # Get corresponding value of modeled data on that date
    value_mod_at_max_obs = combined_df.loc[date_obs_max, 'NWM_HRS_swe_m']
  </code></pre>
</div>


Compare the average melt rate over the full melt period. 

The following function computes the melt period length by identifying the first date after the peak SWE when SWE drops to zero and remains at zero for at least (`min_zero_days`) consecutive days. This is used to define the end of the melt period. Finally, the function calculates the average melt rate, which represents the rate at which snow disappeared, expressed in meters per day, over the full melt period.

In [ ]:
# def compute_melt_period(swe_series, min_zero_days=10):
#     peak_date = swe_series.idxmax()
#     peak_swe = swe_series.max()

#     after_peak = swe_series.loc[peak_date:]

#     zero_streak = 0
#     melt_end_date = None

#     for date, value in after_peak.items():
#         if value == 0:
#             zero_streak += 1
#         else:
#             zero_streak = 0

#         if zero_streak >= min_zero_days:
#             melt_end_date = date
#             break

#     if melt_end_date is None:
#         raise ValueError("Could not find a period of at least 10 consecutive zero SWE days after the peak.")

#     melt_period_days = (melt_end_date - peak_date).days
#     melt_rate = peak_swe / melt_period_days

#     return {
#         'peak_date': peak_date,
#         'peak_swe_m': peak_swe,
#         'melt_end_date': melt_end_date,
#         'melt_period_days': melt_period_days,
#         'melt_rate_m/d': melt_rate
#     }


# def extract_melt_period_stats(df, min_zero_days=10):
#     result = []

#     # Identify CCSS SWE columns
#     ccss_columns = [col for col in df.columns if col.startswith('CCSS_') and col.endswith('_swe_m')]

#     for wy, group in df.groupby('Water_Year'):
#         for station_col in ccss_columns:
#             # Clean series
#             swe_series = pd.to_numeric(group[station_col], errors='coerce').dropna()

#             # Skip if insufficient data
#             if swe_series.empty or swe_series.max() == 0:
#                 continue

#             try:
#                 # Compute melt period stats
#                 stats = compute_melt_period(swe_series, min_zero_days=min_zero_days)
#                 result.append({
#                     'Water_Year': wy,
#                     'Station': station_col,
#                     'Peak_SWE_Date': stats['peak_date'],
#                     'Peak_SWE_m': stats['peak_swe_m'],
#                     'Melt_End_Date': stats['melt_end_date'],
#                     'Melt_Period_Days': stats['melt_period_days'],
#                     'Melt_Rate_m_per_day': stats['melt_rate_m/d']
#                 })
#             except ValueError:
#                 # If melt period cannot be determined, skip
#                 continue

#     return pd.DataFrame(result)


In [ ]:
import importlib
importlib.reload(utils)

In [ ]:
melt_stats_df = utils.compute_melt_period_statistics(combined_df)
melt_stats_df.head()


In [ ]:
observed_melt_period = nwm_utils.compute_melt_period(combined_df[f'CCSS_{my_site_code}_swe_m'])
observed_melt_period

In [ ]:
modeled_melt_period = nwm_utils.compute_melt_period(combined_df[f'NWM_{my_site_code}_swe_m'])
modeled_melt_period